# Import Dependencies

In [ ]:
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# Dataset Load

In [ ]:
# External Validation Dataset
wustl_df = pd.read_csv("datasets/WUSTL/WUSTL_DATASET.csv")

# Internal Validation Datasets
train_df = pd.read_csv("datasets/CICIOMT/train/ciciomt2024_train_balanced.csv")
val_df = pd.read_csv("datasets/CICIOMT/val/ciciomt2024_validation_balanced.csv")
test_df = pd.read_csv("datasets/CICIOMT/test/ciciomt2024_test_balanced.csv")

# WUSTL Dataset Check

In [ ]:
print("WUSTL Shape: ", wustl_df.shape)
print("WUSTL Columns: ", wustl_df.columns)

for col in wustl_df.columns:
    if "label" in col.lower() or "attack" in col.lower() or "class" in col.lower() or "target" in col.lower():
        print(col)
        print(wustl_df[col].value_counts())

# Dataset Feature Harmonization

In [ ]:
def harmonize_ciciomt(df):
    new_df = pd.DataFrame()

    new_df["duration"] = df["Duration"]
    new_df["rate"] = df["Rate"]

    # packet / byte volume
    new_df["total_bytes"] = df["Tot size"]
    new_df["total_packets"] = df["Number"]

    # packet size statistics
    new_df["avg_packet_size"] = df["AVG"]
    new_df["max_packet_size"] = df["Max"]
    new_df["min_packet_size"] = df["Min"]

    # timing / load
    new_df["inter_packet_time"] = df["IAT"]
    new_df["src_load"] = df["Srate"]
    new_df["dst_load"] = df["Drate"]

    # label
    new_df["label"] = df["binary_label"]

    return new_df


def harmonize_wustl(df):
    new_df = pd.DataFrame()

    new_df["duration"] = df["Dur"]
    new_df["rate"] = df["Rate"]

    # packet / byte volume
    new_df["total_bytes"] = df["TotBytes"]
    new_df["total_packets"] = df["TotPkts"]

    # avoid divide-by-zero
    new_df["avg_packet_size"] = df["TotBytes"] / df["TotPkts"].replace(0, np.nan)

    # packet size statistics
    new_df["max_packet_size"] = df[["sMaxPktSz", "dMaxPktSz"]].max(axis=1)
    new_df["min_packet_size"] = df[["sMinPktSz", "dMinPktSz"]].min(axis=1)

    # timing / load
    new_df["inter_packet_time"] = df[["SIntPkt", "DIntPkt"]].mean(axis=1)
    new_df["src_load"] = df["SrcLoad"]
    new_df["dst_load"] = df["DstLoad"]

    # label
    new_df["label"] = df["Label"]

    return new_df


ciciomt_train_h = harmonize_ciciomt(train_df)
ciciomt_val_h = harmonize_ciciomt(val_df)
ciciomt_test_h = harmonize_ciciomt(test_df)
wustl_h = harmonize_wustl(wustl_df)

# Clean NaN and Infinite Values

In [ ]:
# replace inf with NaN
ciciomt_train_h = ciciomt_train_h.replace([np.inf, -np.inf], np.nan)
ciciomt_val_h = ciciomt_val_h.replace([np.inf, -np.inf], np.nan)
ciciomt_test_h = ciciomt_test_h.replace([np.inf, -np.inf], np.nan)
wustl_h = wustl_h.replace([np.inf, -np.inf], np.nan)

# fill missing values using CICIoMT train median
feature_cols_h = [col for col in ciciomt_train_h.columns if col != "label"]

train_medians = ciciomt_train_h[feature_cols_h].median()

ciciomt_train_h[feature_cols_h] = ciciomt_train_h[feature_cols_h].fillna(train_medians)
ciciomt_val_h[feature_cols_h] = ciciomt_val_h[feature_cols_h].fillna(train_medians)
ciciomt_test_h[feature_cols_h] = ciciomt_test_h[feature_cols_h].fillna(train_medians)
wustl_h[feature_cols_h] = wustl_h[feature_cols_h].fillna(train_medians)

print(ciciomt_train_h.columns.tolist())
print(wustl_h.columns.tolist())

print("CICIoMT train:", ciciomt_train_h.shape)
print("CICIoMT val:", ciciomt_val_h.shape)
print("CICIoMT test:", ciciomt_test_h.shape)
print("WUSTL:", wustl_h.shape)

print("CICIoMT label distribution:")
print(ciciomt_train_h["label"].value_counts())

print("WUSTL label distribution:")
print(wustl_h["label"].value_counts())

# Feature and Target Variables

In [ ]:
feature_cols_h = [
    "duration",
    "rate",
    "total_bytes",
    "total_packets",
    "avg_packet_size",
    "max_packet_size",
    "min_packet_size",
    "inter_packet_time",
    "src_load",
    "dst_load"
]

X_train_h = ciciomt_train_h[feature_cols_h]
y_train_h = ciciomt_train_h["label"]

X_val_h = ciciomt_val_h[feature_cols_h]
y_val_h = ciciomt_val_h["label"]

X_test_h = ciciomt_test_h[feature_cols_h]
y_test_h = ciciomt_test_h["label"]

X_wustl_h = wustl_h[feature_cols_h]
y_wustl_h = wustl_h["label"]

# XGBoost Training

In [ ]:
xgb_h = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss"
)

xgb_h.fit(X_train_h, y_train_h)


y_val_pred_h = xgb_h.predict(X_val_h)
y_val_prob_h = xgb_h.predict_proba(X_val_h)[:, 1]
print("CICIoMT Harmonized Validation Metrics:")
print("Accuracy:", accuracy_score(y_val_h, y_val_pred_h))
print("Precision:", precision_score(y_val_h, y_val_pred_h))
print("Recall:", recall_score(y_val_h, y_val_pred_h))
print("F1:", f1_score(y_val_h, y_val_pred_h))
print("ROC-AUC:", roc_auc_score(y_val_h, y_val_prob_h))
print("Confusion Matrix:")
print(confusion_matrix(y_val_h, y_val_pred_h))

y_test_pred_h = xgb_h.predict(X_test_h)
y_test_prob_h = xgb_h.predict_proba(X_test_h)[:, 1]
print("\nCICIoMT Harmonized Internal Test Metrics:")
print("Accuracy:", accuracy_score(y_test_h, y_test_pred_h))
print("Precision:", precision_score(y_test_h, y_test_pred_h))
print("Recall:", recall_score(y_test_h, y_test_pred_h))
print("F1:", f1_score(y_test_h, y_test_pred_h))
print("ROC-AUC:", roc_auc_score(y_test_h, y_test_prob_h))
print("Confusion Matrix:")
print(confusion_matrix(y_test_h, y_test_pred_h))


y_wustl_pred_h = xgb_h.predict(X_wustl_h)
y_wustl_prob_h = xgb_h.predict_proba(X_wustl_h)[:, 1]
print("\nWUSTL External Test Metrics:")
print("Accuracy:", accuracy_score(y_wustl_h, y_wustl_pred_h))
print("Precision:", precision_score(y_wustl_h, y_wustl_pred_h))
print("Recall:", recall_score(y_wustl_h, y_wustl_pred_h))
print("F1:", f1_score(y_wustl_h, y_wustl_pred_h))
print("ROC-AUC:", roc_auc_score(y_wustl_h, y_wustl_prob_h))
print("Confusion Matrix:")
print(confusion_matrix(y_wustl_h, y_wustl_pred_h))

# Using Balanced WUSTL Test

In [ ]:
wustl_normal = wustl_h[wustl_h["label"] == 0]
wustl_attack = wustl_h[wustl_h["label"] == 1]

wustl_normal_sample = wustl_normal.sample(n=len(wustl_attack), random_state=42)

wustl_balanced = pd.concat([wustl_normal_sample, wustl_attack])
wustl_balanced = wustl_balanced.sample(frac=1, random_state=42)

X_wustl_bal = wustl_balanced[feature_cols_h]
y_wustl_bal = wustl_balanced["label"]

y_wustl_bal_pred = xgb_h.predict(X_wustl_bal)
y_wustl_bal_prob = xgb_h.predict_proba(X_wustl_bal)[:, 1]

print("WUSTL Balanced External Test Metrics:")
print("Accuracy:", accuracy_score(y_wustl_bal, y_wustl_bal_pred))
print("Precision:", precision_score(y_wustl_bal, y_wustl_bal_pred))
print("Recall:", recall_score(y_wustl_bal, y_wustl_bal_pred))
print("F1:", f1_score(y_wustl_bal, y_wustl_bal_pred))
print("ROC-AUC:", roc_auc_score(y_wustl_bal, y_wustl_bal_prob))
print("Confusion Matrix:")
print(confusion_matrix(y_wustl_bal, y_wustl_bal_pred))

# WUSTL In-Domain Experiment

In [ ]:
X_wustl = wustl_h[feature_cols_h]
y_wustl = wustl_h["label"]

X_w_train, X_w_test, y_w_train, y_w_test = train_test_split(
    X_wustl,
    y_wustl,
    test_size=0.30,
    random_state=42,
    stratify=y_wustl
)

xgb_wustl = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss"
)

xgb_wustl.fit(X_w_train, y_w_train)

y_w_pred = xgb_wustl.predict(X_w_test)
y_w_prob = xgb_wustl.predict_proba(X_w_test)[:, 1]

print("WUSTL In-Domain Test Metrics:")
print("Accuracy:", accuracy_score(y_w_test, y_w_pred))
print("Precision:", precision_score(y_w_test, y_w_pred))
print("Recall:", recall_score(y_w_test, y_w_pred))
print("F1:", f1_score(y_w_test, y_w_pred))
print("ROC-AUC:", roc_auc_score(y_w_test, y_w_prob))
print("Confusion Matrix:")
print(confusion_matrix(y_w_test, y_w_pred))

# Weighted XGBoost In-Domain

In [ ]:
# Features and label
X_wustl = wustl_h[feature_cols_h]
y_wustl = wustl_h["label"]

# Train-test split
X_w_train, X_w_test, y_w_train, y_w_test = train_test_split(
    X_wustl,
    y_wustl,
    test_size=0.30,
    random_state=42,
    stratify=y_wustl
)

# Calculate imbalance weight
normal_count = (y_w_train == 0).sum()
attack_count = (y_w_train == 1).sum()

scale_pos_weight = normal_count / attack_count

xgb_wustl_weighted = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss"
)

xgb_wustl_weighted.fit(X_w_train, y_w_train)

y_w_pred_weighted = xgb_wustl_weighted.predict(X_w_test)
y_w_prob_weighted = xgb_wustl_weighted.predict_proba(X_w_test)[:, 1]

print("WUSTL Weighted XGBoost In-Domain Test Metrics:")
print("Accuracy:", accuracy_score(y_w_test, y_w_pred_weighted))
print("Precision:", precision_score(y_w_test, y_w_pred_weighted))
print("Recall:", recall_score(y_w_test, y_w_pred_weighted))
print("F1:", f1_score(y_w_test, y_w_pred_weighted))
print("ROC-AUC:", roc_auc_score(y_w_test, y_w_prob_weighted))
print("Confusion Matrix:")
print(confusion_matrix(y_w_test, y_w_pred_weighted))

# SHAP for WUSTL

In [ ]:
# Use all WUSTL test samples, because the test set is not too large
X_wustl_shap = X_w_test.copy()

print("SHAP sample shape:", X_wustl_shap.shape)

# Run SHAP
explainer_wustl = shap.TreeExplainer(xgb_wustl)
shap_values_wustl = explainer_wustl.shap_values(X_wustl_shap)

# Create SHAP importance table
shap_importance_wustl = pd.DataFrame({
    "feature": X_wustl_shap.columns,
    "mean_abs_shap": np.abs(shap_values_wustl).mean(axis=0)
})

shap_importance_wustl = shap_importance_wustl.sort_values(
    by="mean_abs_shap",
    ascending=False
)

print("WUSTL SHAP Feature Importance:")
print(shap_importance_wustl)


plt.figure()
shap.summary_plot(
    shap_values_wustl,
    X_wustl_shap,
    plot_type="bar",
    show=False
)
plt.tight_layout()
plt.show()

plt.figure()
shap.summary_plot(
    shap_values_wustl,
    X_wustl_shap,
    show=False
)
plt.tight_layout()
plt.show()

# WUSTL Behavior-Group SHAP Contribution

In [ ]:
harmonized_feature_groups = {
    "duration": "Timing/rate behavior",
    "rate": "Timing/rate behavior",
    "inter_packet_time": "Timing/rate behavior",

    "total_bytes": "Packet-size/volume behavior",
    "total_packets": "Packet-size/volume behavior",
    "avg_packet_size": "Packet-size/volume behavior",
    "max_packet_size": "Packet-size/volume behavior",
    "min_packet_size": "Packet-size/volume behavior",

    "src_load": "Load behavior",
    "dst_load": "Load behavior"
}

shap_importance_wustl["behavior_group"] = shap_importance_wustl["feature"].map(
    harmonized_feature_groups
)

print("Missing group features:")
print(shap_importance_wustl[shap_importance_wustl["behavior_group"].isna()])

wustl_group_contribution = (
    shap_importance_wustl
    .groupby("behavior_group")["mean_abs_shap"]
    .sum()
    .reset_index()
)

total_shap = wustl_group_contribution["mean_abs_shap"].sum()

wustl_group_contribution["percentage"] = (
    wustl_group_contribution["mean_abs_shap"] / total_shap * 100
)

wustl_group_contribution = wustl_group_contribution.sort_values(
    by="percentage",
    ascending=False
)

print("WUSTL Behavior-Group SHAP Contribution:")
print(wustl_group_contribution)


# Plot
plt.figure(figsize=(8, 5))
plt.barh(
    wustl_group_contribution["behavior_group"],
    wustl_group_contribution["percentage"]
)
plt.gca().invert_yaxis()
plt.xlabel("SHAP Contribution (%)")
plt.ylabel("Behavior Group")
plt.title("WUSTL In-Domain Behavior-Group SHAP Contribution")
plt.tight_layout()
plt.show()